# Clinical NER Evaluation — Colab Runner (T4 free tier)

Benchmarks Bio_ClinicalBERT, BioBERT, and PubMedBERT on a combined
**NCBI Disease + BC5CDR + MACCROBAT** NER task (harmonized to Disease/Chemical),
then produces a single comparison table.

**Before running:** Runtime → Change runtime type → **T4 GPU**.
Checkpoints are written to Google Drive so a disconnect costs at most the
current model, not the whole sweep.

In [ ]:
# 1. Mount Drive for durable checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_ROOT = '/content/drive/MyDrive/clinical-ner-eval/checkpoints'
os.makedirs(CKPT_ROOT, exist_ok=True)

In [ ]:
# 2. Get the project code and install deps with uv
# Replace with your repo URL, or upload the clinical-ner-eval folder to Drive.
!pip install -q uv
# !git clone <YOUR_REPO_URL> clinical-ner-eval
%cd clinical-ner-eval
!uv pip install --system -q transformers torch datasets seqeval evaluate accelerate pandas

In [ ]:
# 3. Point the harness at the Drive checkpoint dir, confirm GPU
import src.config as config
config.CHECKPOINT_DIR = CKPT_ROOT

import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Train sweep — loops over all shortlisted models, checkpointing each to Drive.
# Resumable: a model whose checkpoint already exists in Drive is skipped.
from src.train import train_model
from src.config import MODELS

for key in MODELS:
    out = os.path.join(CKPT_ROOT, key)
    if os.path.isdir(out) and os.listdir(out):
        print(f'[skip] {key} already trained')
        continue
    print(f'\n=== training {key} ===')
    train_model(key)

In [ ]:
# 5. Evaluate all checkpoints and print the comparison table
from src.evaluate import main as eval_main
eval_main()

## Optional: MACCROBAT-only rich-entity showcase

Demonstrates the full note→entity extraction with all native MACCROBAT types
(not collapsed to Disease/Chemical) — useful to show the manager the
"doctor notes → expected entities" flow on real clinical text.

In [ ]:
train_model('bio_clinicalbert', dataset_names=['maccrobat'], label_mode='native')